In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# === 어댑터즈 챗봇이 누적해 온 가상의 대화 이력 ===
messages = [
    SystemMessage(content="당신은 어댑터즈 안내 담당자입니다."),
    HumanMessage(content="어댑터즈는 무슨 서비스인가요?"),
    AIMessage(content="어댑터즈는 스타트업코드의 개발 교재 서빙 서비스입니다."),
    HumanMessage(content="어떤 분야의 교재가 있나요?"),
    AIMessage(content="인공지능, 데이터 분석, 웹 개발, 인프라 분야가 있습니다."),
    HumanMessage(content="교수법은 뭔가요?"),
    AIMessage(content="5단 분석법이라는 독자적인 교수법을 사용합니다."),
    HumanMessage(content="5단 분석법이 뭔가요?"),
]

print(f"전체 메시지 수: {len(messages)}")

In [ ]:
# 개수 기준 트리밍
from langchain_core.messages import trim_messages

# === 최근 4개 메시지만 유지 (System 메시지 별도 보존) ===
trimmed = trim_messages(
    messages,
    max_tokens=4,                # 4개 메시지까지 유지
    strategy="last",             # 최근부터 유지
    token_counter=len,           # 메시지 개수 기준
    include_system=True,         # System 메시지는 항상 유지
    start_on="human",            # 결과의 첫 일반 메시지는 HumanMessage
)

print(f"잘라낸 후 메시지 수: {len(trimmed)}")
for msg in trimmed:
    print(f"{type(msg).__name__}: {msg.content}")

In [ ]:
# LCEL 체인에 자동 트리밍 끼워 넣기
from operator import itemgetter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# === Chat Model ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key="YOUR_API_KEY"
)

# === 트리머 정의: 최근 4개 메시지 + System ===
trimmer = trim_messages(
    max_tokens=4,
    strategy="last",
    token_counter=len,
    include_system=True,
    start_on="human",
)

# === 프롬프트 ===
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 어댑터즈 안내 담당자입니다. "
               "어댑터즈는 스타트업코드의 개발 교재 서빙 서비스입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# === 체인: history를 trim한 뒤 프롬프트에 주입 ===
chain = (
    RunnablePassthrough.assign(
        history=itemgetter("history") | trimmer
    )
    | prompt | model | StrOutputParser()
)

In [ ]:
# RunnableWithMessageHistory와 결합
import time
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# === 세션별 이력 저장소 ===
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# === 트리밍이 포함된 chain을 그대로 RunnableWithMessageHistory로 감싸기 ===
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# === 여러 번 호출하며 이력을 쌓아 보기 ===
config = {"configurable": {"session_id": "user-001"}}

questions = [
    "어댑터즈는 무슨 서비스인가요?",
    "어떤 분야의 교재가 있나요?",
    "그 중에서 어떤 분야가 가장 인기 있나요?"   # 트리밍 후에도 직전 대화 맥락은 살아 있어야 함
]

for q in questions:
    response = chain_with_history.invoke({"input": q}, config=config)
    print(f"Q: {q}")
    print(f"A: {response}\n")
    # === 무료 티어 분당 한도 회피용 딜레이 ===
    time.sleep(7)